In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q flask pyngrok faiss-cpu sentence-transformers transformers accelerate pypdf

In [ ]:
import io, re, json, uuid, math, threading, numpy as np, faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
DOCS = {}          # doc_id -> {"chunks", "index", "cards"}
JOBS = {}          # job_id -> progress/result of a background generation
MODEL_LOCK = threading.Lock()   # embedder + LLM are shared, so one caller at a time
MAX_MB = 50

def pdf_to_text(file_bytes, start=None, end=None):
    reader = PdfReader(io.BytesIO(file_bytes))
    total = len(reader.pages)
    s, e = max(1, start or 1), min(total, end or total)
    if s > e:
        raise ValueError(f"Page range {s}-{e} is invalid (PDF has {total} pages).")
    return "\n".join(reader.pages[p].extract_text() or "" for p in range(s - 1, e))

def chunk_text(text, size=1500, overlap=200):
    text = re.sub(r"\s+", " ", text).strip()
    return [text[i:i+size] for i in range(0, len(text), size - overlap) if text[i:i+size].strip()]

def ingest(file_bytes, start=None, end=None):
    if len(file_bytes) > MAX_MB * 1024 * 1024:
        raise ValueError(f"PDF is larger than {MAX_MB} MB.")
    chunks = chunk_text(pdf_to_text(file_bytes, start, end))
    if not chunks:
        raise ValueError("No text found in this PDF (or page range). It may be a scanned image that needs OCR.")
    with MODEL_LOCK:
        emb = embedder.encode(chunks, normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    doc_id = uuid.uuid4().hex[:8]
    DOCS[doc_id] = {"chunks": chunks, "index": index, "cards": []}
    return doc_id, len(chunks)

In [ ]:
import torch
from transformers import pipeline

llm = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct",
               torch_dtype=torch.float16, device_map="auto")

CARD_PROMPT = """You are a study assistant. From the text below, write {n} flashcards.
Each card tests one important fact or concept. Questions must be self-contained.
Answers must be short (1-2 sentences) and based only on the text.
Return ONLY a JSON list like: [{{"q": "...", "a": "..."}}]

Text:
{text}"""

STRICT_PROMPT = """Write {n} flashcards from the text below.
Output ONLY valid JSON: a list of objects with exactly the keys "q" and "a".
Use double quotes, no trailing commas, no markdown, no commentary. Do not use square brackets or curly braces inside the strings.

Text:
{text}"""

def llm_chat(prompt, max_new_tokens):
    with MODEL_LOCK:
        out = llm([{"role": "user", "content": prompt}], max_new_tokens=max_new_tokens,
                  do_sample=False)[0]["generated_text"][-1]["content"]
    return out

def parse_cards(raw):
    """Pull out each {...} object separately, so one malformed card (or a stray bracket) doesn't lose the rest."""
    cards = []
    for m in re.finditer(r"\{[^{}]*\}", raw, re.DOTALL):
        try:
            c = json.loads(m.group(0))
        except json.JSONDecodeError:
            continue
        if isinstance(c.get("q"), str) and isinstance(c.get("a"), str) and c["q"].strip() and c["a"].strip():
            cards.append({"q": c["q"].strip(), "a": c["a"].strip()})
    return cards

def evenly(items, k):
    """Pick k items spread evenly across the list so the whole PDF stays covered."""
    if k >= len(items): return list(items)
    idx = sorted(set(np.linspace(0, len(items) - 1, k).round().astype(int)))
    return [items[i] for i in idx]

def pick_chunks(doc, target, topic=None):
    chunks = doc["chunks"]
    if topic and topic.strip():   # use the FAISS index: only the chunks most relevant to the topic
        with MODEL_LOCK:
            qv = embedder.encode([topic.strip()], normalize_embeddings=True).astype("float32")
        k = min(len(chunks), max(3, math.ceil(target / 3)))
        _, ids = doc["index"].search(qv, k)
        return [chunks[i] for i in sorted(int(i) for i in ids[0] if i >= 0)]
    return evenly(chunks, target)

def generate_cards(doc_id, target=30, topic=None, job=None, dup_threshold=0.85):
    doc = DOCS[doc_id]
    use = pick_chunks(doc, target, topic)
    per_chunk = max(1, math.ceil(target * 1.3 / len(use)))   # ask for ~30% extra, dedupe removes near-duplicates
    cards, vecs = [], []
    for n_done, ch in enumerate(use, 1):
        new = parse_cards(llm_chat(CARD_PROMPT.format(n=per_chunk, text=ch), 120 * per_chunk + 100))
        if not new:   # malformed output: one retry with a stricter prompt
            new = parse_cards(llm_chat(STRICT_PROMPT.format(n=per_chunk, text=ch), 120 * per_chunk + 100))
        if new:
            with MODEL_LOCK:
                nv = embedder.encode([c["q"] for c in new], normalize_embeddings=True)
            for c, v in zip(new, nv):
                if vecs and float(np.max(np.stack(vecs) @ v)) >= dup_threshold:
                    continue   # near-duplicate question (semantic, not just exact text)
                vecs.append(v); cards.append(c)
        if job is not None:
            job.update(done=n_done, total=len(use), found=len(cards))
    cards = evenly(cards, target)
    doc["cards"] = cards
    return cards

def grade(student_answer, reference, question=""):
    """Returns (similarity_score, is_correct). Similarity alone is never trusted in the middle band."""
    if not student_answer.strip():
        return 0.0, False
    with MODEL_LOCK:
        v = embedder.encode([student_answer, reference], normalize_embeddings=True)
    score = float(v[0] @ v[1])
    if score >= 0.90: return score, True       # near-identical wording
    if score < 0.35:  return score, False      # clearly unrelated
    # Everything in between: similar-sounding answers can still be factually wrong (e.g. "increases" vs "decreases")
    prompt = f"""Question: {question}
Reference answer: {reference}
Student answer: {student_answer}

Does the student's answer convey the same key idea as the reference answer? Be lenient about wording, but strict about facts and direction (increase/decrease, true/false, before/after). Reply with only CORRECT or INCORRECT."""
    out = llm_chat(prompt, 5)
    return score, out.strip().upper().startswith("CORRECT")

In [ ]:
PAGE = r"""<!DOCTYPE html>
<html lang="en"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Study Buddy</title>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<style>
:root{--bg:#f4f5fb;--surface:#fff;--text:#1d2130;--muted:#6b7280;--border:#e4e6ef;--primary:#6366f1;--primary-d:#4f46e5;
  --soft:#eef0ff;--good:#10b981;--good-soft:#d9f7ec;--bad:#ef4444;--bad-soft:#fde4e4;--warn:#f59e0b;
  --shadow:0 1px 2px rgba(16,24,40,.05),0 10px 30px rgba(16,24,40,.08);--glow:radial-gradient(900px 400px at 15% -10%,#dfe2ff 0,transparent 60%),radial-gradient(700px 400px at 100% 0,#ffe3f1 0,transparent 55%)}
@media (prefers-color-scheme:dark){:root:not([data-theme=light]){--bg:#0f1220;--surface:#171b2e;--text:#e8eaf6;--muted:#9aa1bd;--border:#262b45;--soft:#22274a;
  --good-soft:#0f3a2e;--bad-soft:#4a1d24;--shadow:0 1px 2px rgba(0,0,0,.4),0 10px 30px rgba(0,0,0,.35);--glow:radial-gradient(900px 400px at 15% -10%,#25295a 0,transparent 60%),radial-gradient(700px 400px at 100% 0,#3a1f3d 0,transparent 55%)}}
:root[data-theme=dark]{--bg:#0f1220;--surface:#171b2e;--text:#e8eaf6;--muted:#9aa1bd;--border:#262b45;--soft:#22274a;
  --good-soft:#0f3a2e;--bad-soft:#4a1d24;--shadow:0 1px 2px rgba(0,0,0,.4),0 10px 30px rgba(0,0,0,.35);--glow:radial-gradient(900px 400px at 15% -10%,#25295a 0,transparent 60%),radial-gradient(700px 400px at 100% 0,#3a1f3d 0,transparent 55%)}
*{box-sizing:border-box}
body{margin:0;min-height:100vh;font-family:Inter,system-ui,-apple-system,Segoe UI,sans-serif;color:var(--text);background:var(--bg) var(--glow) no-repeat;padding:22px 14px 60px;line-height:1.45}
.wrap{max-width:720px;margin:0 auto}
.top{display:flex;align-items:center;justify-content:space-between;gap:12px;margin-bottom:14px}
.brand{display:flex;align-items:center;gap:12px}
.logo{display:grid;place-items:center;width:46px;height:46px;border-radius:14px;background:linear-gradient(135deg,var(--primary),#ec4899);font-size:1.5rem;box-shadow:var(--shadow)}
h1{margin:0;font-size:1.35rem;font-weight:800;letter-spacing:-.02em}
.brand p{margin:0;color:var(--muted);font-size:.85rem}
h2,h3{margin:0 0 8px;letter-spacing:-.01em}
.steps{display:flex;gap:6px;margin:0 0 16px;padding:4px;background:var(--surface);border:1px solid var(--border);border-radius:999px;box-shadow:var(--shadow)}
.steps span{flex:1;text-align:center;padding:7px 8px;border-radius:999px;font-size:.82rem;font-weight:600;color:var(--muted);transition:.2s}
.steps span.on{background:var(--primary);color:#fff}
.panel{background:var(--surface);border:1px solid var(--border);border-radius:20px;padding:22px;box-shadow:var(--shadow);margin-bottom:14px}
.hidden{display:none!important}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.between{justify-content:space-between}.center{justify-content:center}
.muted{color:var(--muted)}.small{font-size:.85rem}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:12px}
@media(max-width:560px){.grid2{grid-template-columns:1fr}}
.field label,.field .lbl{display:block;font-size:.78rem;font-weight:700;color:var(--muted);margin-bottom:5px;text-transform:uppercase;letter-spacing:.04em}

button{font:inherit;font-weight:600;border:1px solid transparent;border-radius:12px;padding:10px 16px;cursor:pointer;color:#fff;background:var(--primary);
  transition:transform .12s,background .15s,box-shadow .15s}
button:hover{background:var(--primary-d);transform:translateY(-1px)}
button:active{transform:translateY(1px)}
button:disabled{opacity:.5;cursor:not-allowed;transform:none}
button.ghost{background:transparent;color:var(--text);border-color:var(--border)}
button.ghost:hover{background:var(--soft)}
button.good{background:var(--good)}button.bad{background:var(--bad)}
button.lg{padding:13px 22px;font-size:1rem;border-radius:14px}
button.icon{padding:9px 12px;background:var(--surface);color:var(--text);border-color:var(--border)}
:focus-visible{outline:3px solid var(--primary);outline-offset:2px}
input[type=text],input[type=number]{width:100%;font:inherit;color:var(--text);background:var(--bg);border:1.5px solid var(--border);border-radius:12px;padding:10px 12px;outline:none}
input:focus{border-color:var(--primary)}
input[type=range]{width:100%;accent-color:var(--primary)}

.drop{display:flex;flex-direction:column;align-items:center;gap:6px;border:2px dashed var(--primary);border-radius:16px;padding:30px 16px;text-align:center;
  cursor:pointer;background:var(--soft);margin-bottom:16px;transition:.15s}
.drop.over{transform:scale(1.01);background:var(--bg)}
.drop .big{font-size:2rem}.drop b{font-size:1.02rem}
.bar{height:10px;background:var(--border);border-radius:99px;overflow:hidden}
.bar i{display:block;height:100%;width:0;background:linear-gradient(90deg,var(--primary),#ec4899);border-radius:99px;transition:width .35s}
.bar.good i{background:var(--good)}
.bar.timer i{background:var(--primary);transition:width .1s linear}.bar.timer.low i{background:var(--bad)}
#status{margin-top:14px;text-align:center;font-weight:600;min-height:24px}
.spin{display:inline-block;animation:spin 1.2s linear infinite}@keyframes spin{to{transform:rotate(360deg)}}
.chip{padding:6px 12px;border-radius:999px;font-size:.82rem;font-weight:600;background:var(--soft);color:var(--text);border:1px solid var(--border);cursor:pointer}
.chip.on{background:var(--primary);color:#fff;border-color:var(--primary)}
.pill{padding:5px 12px;border-radius:999px;font-size:.82rem;font-weight:700;background:var(--soft)}
.pill.g{background:var(--good-soft);color:var(--good)}.pill.r{background:var(--bad-soft);color:var(--bad)}.pill.y{background:#fff1cf;color:#a76a00}
@media (prefers-color-scheme:dark){:root:not([data-theme=light]) .pill.y{background:#4a3a12;color:#fbbf24}}
:root[data-theme=dark] .pill.y{background:#4a3a12;color:#fbbf24}

.scene{perspective:1200px;cursor:pointer;margin:16px 0 8px;user-select:none}
.flipper{position:relative;height:260px;transition:transform .55s cubic-bezier(.3,.7,.2,1);transform-style:preserve-3d}
.flipper.flipped{transform:rotateY(180deg)}
.face{position:absolute;inset:0;border-radius:18px;backface-visibility:hidden;-webkit-backface-visibility:hidden;display:flex;flex-direction:column;
  align-items:center;justify-content:center;padding:26px;text-align:center;overflow:auto;box-shadow:var(--shadow)}
.face small{letter-spacing:.18em;font-weight:700;font-size:.7rem;opacity:.85;margin-bottom:10px}
.face p{margin:0;font-size:1.25rem;font-weight:600;line-height:1.4}
.front{background:linear-gradient(135deg,#6366f1,#8b5cf6);color:#fff}
.back{background:linear-gradient(135deg,#059669,#10b981);color:#fff;transform:rotateY(180deg)}
.keys{display:flex;justify-content:center;gap:14px;flex-wrap:wrap;font-size:.78rem;color:var(--muted);margin:6px 0 14px}
kbd{background:var(--soft);border:1px solid var(--border);border-bottom-width:2px;border-radius:6px;padding:1px 6px;font:600 .75rem Inter,sans-serif;color:var(--text)}

#qText{font-size:1.3rem;margin:18px 0 14px;line-height:1.35}
.timerRow{display:flex;align-items:center;gap:12px;margin-top:10px}.timerRow .bar{flex:1}#timerText{font-weight:700;min-width:62px;font-variant-numeric:tabular-nums}
.feedback{margin-top:16px;padding:16px;border-radius:14px;animation:pop .25s}
.feedback.good{background:var(--good-soft);border:1px solid var(--good)}.feedback.bad{background:var(--bad-soft);border:1px solid var(--bad)}
.feedback .cmp{margin:8px 0;font-size:.95rem}.feedback .cmp b{display:block;font-size:.72rem;text-transform:uppercase;letter-spacing:.05em;opacity:.7}
@keyframes pop{from{transform:translateY(6px);opacity:0}to{transform:none;opacity:1}}

.ringWrap{display:grid;place-items:center;margin:6px 0 4px}
.ring{position:relative;width:140px;height:140px}.ring svg{transform:rotate(-90deg)}
.ring .num{position:absolute;inset:0;display:grid;place-items:center;font-size:1.9rem;font-weight:800}
.stats{display:grid;grid-template-columns:repeat(auto-fit,minmax(120px,1fr));gap:10px;margin:16px 0}
.stat{background:var(--bg);border:1px solid var(--border);border-radius:14px;padding:12px;text-align:center}
.stat b{display:block;font-size:1.25rem}.stat small{color:var(--muted);font-weight:600}
ul.missed{list-style:none;padding:0;margin:0;text-align:left}
ul.missed li{padding:10px 12px;border:1px solid var(--border);border-left:4px solid var(--bad);border-radius:10px;margin-bottom:8px;background:var(--bg)}
ul.missed span{color:var(--muted);font-size:.9rem}
@media (prefers-reduced-motion:reduce){*{animation:none!important;transition:none!important}}

/* ===== COLOUR PACK ===== */
:root{--rainbow:linear-gradient(90deg,#ff6b6b,#feca57,#1dd1a1,#48dbfb,#5f27cd,#ff9ff3);
  --g-primary:linear-gradient(135deg,#7c3aed,#ec4899);--g-good:linear-gradient(135deg,#10b981,#06b6d4);
  --g-bad:linear-gradient(135deg,#ef4444,#f97316);--g-warm:linear-gradient(135deg,#f59e0b,#ef4444);
  --bg-grad:linear-gradient(135deg,#ffd6e8 0%,#e0d4ff 30%,#c9f0ff 60%,#d4ffe4 100%)}
@media (prefers-color-scheme:dark){:root:not([data-theme=light]){--bg-grad:linear-gradient(135deg,#2b1055 0%,#1b2a6b 50%,#0f4c5c 100%)}}
:root[data-theme=dark]{--bg-grad:linear-gradient(135deg,#2b1055 0%,#1b2a6b 50%,#0f4c5c 100%)}
body{background:var(--bg-grad) fixed!important;background-size:200% 200%!important;animation:bgmove 20s ease infinite}
@keyframes bgmove{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}
@keyframes floaty{0%,100%{transform:translateY(0) rotate(-4deg)}50%{transform:translateY(-6px) rotate(4deg)}}
@keyframes fall{to{transform:translateY(110vh) rotate(720deg);opacity:.9}}

h1{background:var(--g-primary);-webkit-background-clip:text;background-clip:text;color:transparent;font-size:1.7rem}
.logo{background:var(--g-primary);animation:floaty 3s ease-in-out infinite}
.steps{background:rgba(255,255,255,.55);backdrop-filter:blur(8px)}
.steps span.on{background:var(--g-primary);box-shadow:0 4px 14px rgba(124,58,237,.45)}
.steps span:nth-child(2).on{background:var(--g-warm)}.steps span:nth-child(3).on{background:var(--g-good)}

.panel{position:relative;overflow:hidden;border:0;background:var(--surface)}
.panel::before{content:"";position:absolute;top:0;left:0;right:0;height:6px;background:var(--rainbow)}

button{background:var(--g-primary);box-shadow:0 4px 14px rgba(124,58,237,.35)}
button:hover{background:var(--g-primary);filter:brightness(1.1);box-shadow:0 6px 18px rgba(124,58,237,.5)}
button.good{background:var(--g-good);box-shadow:0 4px 14px rgba(16,185,129,.4)}
button.bad{background:var(--g-bad);box-shadow:0 4px 14px rgba(239,68,68,.4)}
button.ghost{background:transparent;box-shadow:none;border:2px solid transparent;
  background-image:linear-gradient(var(--surface),var(--surface)),var(--g-primary);background-origin:border-box;background-clip:padding-box,border-box;color:var(--text)}
button.ghost:hover{filter:none;background-image:linear-gradient(var(--soft),var(--soft)),var(--g-primary)}
button.icon{background:var(--surface);box-shadow:none;border:1px solid var(--border)}

.drop{border:3px dashed #ec4899;background:linear-gradient(135deg,rgba(124,58,237,.10),rgba(236,72,153,.10),rgba(6,182,212,.10))}
.drop .big{font-size:2.6rem;animation:floaty 2.5s ease-in-out infinite}
.drop:hover{border-color:#7c3aed}

.chip{border:2px solid transparent}
.chip.on{background:var(--g-primary);box-shadow:0 3px 10px rgba(124,58,237,.4)}
.pill{background:linear-gradient(135deg,#ede9fe,#fce7f3);color:#6d28d9}
.pill.g{background:linear-gradient(135deg,#d1fae5,#cffafe);color:#047857}
.pill.r{background:linear-gradient(135deg,#fee2e2,#ffedd5);color:#b91c1c}
.pill.y{background:linear-gradient(135deg,#fef3c7,#ffedd5);color:#b45309}

.bar{height:12px}.bar i{background:var(--rainbow)}
.bar.good i{background:var(--g-good)}.bar.timer i{background:linear-gradient(90deg,#06b6d4,#7c3aed)}.bar.timer.low i{background:var(--g-bad)}
input[type=range]{accent-color:#ec4899}
input:focus{border-color:#ec4899!important;box-shadow:0 0 0 4px rgba(236,72,153,.18)}

.face{box-shadow:0 12px 30px rgba(0,0,0,.22)}
.face p{font-size:1.35rem;font-weight:700;text-shadow:0 2px 6px rgba(0,0,0,.2)}
.face::after{content:"";position:absolute;inset:0;border-radius:18px;pointer-events:none;
  background:radial-gradient(circle at 15% 10%,rgba(255,255,255,.35),transparent 45%)}

.feedback.good{background:linear-gradient(135deg,#d1fae5,#cffafe);border-color:#10b981;color:#064e3b}
.feedback.bad{background:linear-gradient(135deg,#fee2e2,#ffedd5);border-color:#ef4444;color:#7f1d1d}

.stat{border:0;color:#fff}
.stat small{color:rgba(255,255,255,.9)}
.stat:nth-child(1){background:linear-gradient(135deg,#7c3aed,#a78bfa)}
.stat:nth-child(2){background:linear-gradient(135deg,#06b6d4,#3b82f6)}
.stat:nth-child(3){background:linear-gradient(135deg,#f59e0b,#f97316)}
.stat:nth-child(4){background:linear-gradient(135deg,#ec4899,#ef4444)}
.ring .num{background:var(--g-primary);-webkit-background-clip:text;background-clip:text;color:transparent;font-size:2.2rem}
ul.missed li{border-left-width:6px;border-image:linear-gradient(180deg,#ef4444,#f97316) 1;border-left-style:solid}
kbd{background:linear-gradient(135deg,#ede9fe,#fce7f3);color:#6d28d9;border-color:#d8b4fe}

/* fixes for the colour pack */
button.chip{box-shadow:none}
button.chip:hover{background:var(--soft);filter:none;box-shadow:none}
button.chip.on:hover{background:var(--g-primary)}
@media (prefers-color-scheme:dark){:root:not([data-theme=light]) .steps{background:rgba(23,27,46,.65)}}
:root[data-theme=dark] .steps{background:rgba(23,27,46,.65)}
</style></head>
<body><div class="wrap">

<header class="top">
  <div class="brand"><div class="logo">🧠</div><div><h1>Study Buddy</h1><p>Turn any PDF into flashcards, then quiz yourself</p></div></div>
  <button class="icon" id="themeBtn" aria-label="Toggle dark mode" title="Toggle dark mode">🌓</button>
</header>
<nav class="steps" aria-label="Progress"><span data-s="Upload">1 · Upload</span><span data-s="Study">2 · Study</span><span data-s="Quiz">3 · Quiz</span></nav>

<!-- UPLOAD -->
<section id="scrUpload"><div class="panel">
  <label class="drop" id="drop" for="pdf">
    <span class="big">📄</span><b id="fname">Drop a PDF here or click to browse</b>
    <span class="muted small" id="fmeta">Up to 20 MB · text-based PDFs work best</span>
    <input type="file" id="pdf" accept=".pdf,application/pdf" hidden>
  </label>
  <div class="grid2" style="margin-bottom:14px">
    <div class="field"><label for="topic">🎯 Focus topic (optional)</label><input type="text" id="topic" placeholder="e.g. backpropagation"></div>
    <div class="field"><span class="lbl">📑 Page range (optional)</span>
      <div class="row" style="flex-wrap:nowrap"><input type="number" id="pgStart" placeholder="from" min="1"><span class="muted">–</span><input type="number" id="pgEnd" placeholder="to" min="1"></div></div>
  </div>
  <div class="field" style="margin-bottom:16px"><label for="target">🃏 Number of cards: <span id="targetVal" style="color:var(--primary)">30</span></label>
    <input type="range" id="target" min="5" max="100" step="5" value="30"></div>
  <div class="row between">
    <button class="ghost hidden" id="resumeBtn" onclick="resumeDeck()"></button>
    <button id="makeBtn" class="lg" style="margin-left:auto" onclick="makeCards()">Make my cards ✨</button>
  </div>
  <div class="bar hidden" id="genBar" style="margin-top:16px"><i id="genFill"></i></div>
  <div id="status" role="status" aria-live="polite"></div>
</div></section>

<!-- STUDY -->
<section id="scrStudy" class="hidden">
  <div class="panel">
    <div class="row between">
      <div class="row"><span class="pill" id="sPos"></span><span class="pill g" id="knownPill">✓ 0 known</span></div>
      <div class="row"><button class="chip on" id="fAll" onclick="setFilter('all')">All</button><button class="chip" id="fLearn" onclick="setFilter('learn')">Still learning</button></div>
    </div>
    <div class="bar good" style="margin-top:12px"><i id="studyBar"></i></div>
    <div class="scene" onclick="flipCard()" role="button" tabindex="0" aria-label="Flip card"><div class="flipper" id="flipper">
      <div class="face front"><small>QUESTION</small><p id="sQ"></p></div>
      <div class="face back"><small>ANSWER</small><p id="sA"></p></div>
    </div></div>
    <div class="keys"><span><kbd>Space</kbd> flip</span><span><kbd>←</kbd><kbd>→</kbd> move</span><span><kbd>1</kbd> still learning</span><span><kbd>2</kbd> got it</span></div>
    <div class="row center">
      <button class="ghost" onclick="studyMove(-1)">← Prev</button>
      <button class="bad" onclick="mark(false)">✗ Still learning</button>
      <button class="good" onclick="mark(true)">✓ Got it</button>
      <button class="ghost" onclick="studyMove(1)">Next →</button>
    </div>
    <div class="row center" style="margin-top:14px">
      <button class="ghost" onclick="flipCard()">🔄 Flip</button>
      <button class="ghost" onclick="shuffleStudy()">🔀 Shuffle</button>
      <button class="ghost" onclick="exportCards()">⬇ Export (Anki)</button>
      <button class="ghost" onclick="showScreen('Upload')">📄 New PDF</button>
    </div>
  </div>
  <div class="panel">
    <div class="row between">
      <div><b>⏱ Seconds per card</b><div class="muted small">0 = no time limit</div></div>
      <div class="row"><input type="number" id="limit" value="30" min="0" max="300" style="width:84px">
        <button class="chip" onclick="setLimit(0)">Off</button><button class="chip" onclick="setLimit(15)">15</button><button class="chip" onclick="setLimit(30)">30</button><button class="chip" onclick="setLimit(60)">60</button></div>
    </div>
    <button class="lg" style="width:100%;margin-top:14px" onclick="startQuiz()">Start quiz 🚀</button>
  </div>
</section>

<!-- QUIZ -->
<section id="scrQuiz" class="hidden"><div class="panel">
  <div class="row between">
    <span class="pill" id="qPos"></span>
    <div class="row"><span class="pill g" id="scorePill">✅ 0</span><span class="pill r" id="missPill">❌ 0</span><span class="pill y" id="streakPill">🔥 0</span></div>
  </div>
  <div class="bar good" style="margin-top:12px"><i id="progress"></i></div>
  <div class="timerRow"><span id="timerText">⏱ 30s</span><div class="bar timer" id="timerTrack"><i id="timerBar"></i></div></div>
  <h2 id="qText"></h2>
  <input type="text" id="answer" autocomplete="off" placeholder="Type your answer and press Enter…">
  <div class="row" style="margin-top:12px">
    <button class="good" id="submitBtn" onclick="submitAnswer(false)">Submit ✔</button>
    <button class="ghost" onclick="exitTest()">Exit test</button>
  </div>
  <div id="feedback" class="feedback hidden" aria-live="polite"></div>
</div></section>

<!-- SUMMARY -->
<section id="scrSummary" class="hidden"><div class="panel" style="text-align:center">
  <div class="ringWrap"><div class="ring">
    <svg width="140" height="140" viewBox="0 0 140 140"><circle cx="70" cy="70" r="58" fill="none" stroke="var(--border)" stroke-width="12"/>
      <circle id="ringFg" cx="70" cy="70" r="58" fill="none" stroke="var(--primary)" stroke-width="12" stroke-linecap="round" stroke-dasharray="364.4" stroke-dashoffset="364.4" style="transition:stroke-dashoffset .9s"/></svg>
    <div class="num" id="ringNum">0%</div></div></div>
  <h2 id="sumTitle"></h2><p id="sumNote" class="muted small" style="margin:0"></p>
  <div class="stats">
    <div class="stat"><b id="stScore"></b><small>Score</small></div>
    <div class="stat"><b id="stAvg"></b><small>Avg per card</small></div>
    <div class="stat"><b id="stTotal"></b><small>Total time</small></div>
    <div class="stat"><b id="stStreak"></b><small>Best streak</small></div>
  </div>
  <div id="missedBox" style="text-align:left"></div>
  <div class="row center" id="sumButtons" style="margin-top:16px"></div>
</div></section>
</div>

<script>
let allCards=[], studyCards=[], cards=[], missedCards=[], known=new Set(), filter='all';
let s=0, i=0, results=[], limit=30, t0=0, tick=null, nextTimer=null;
let busy=false, awaitingNext=false, streak=0, bestStreak=0, correctCount=0, sessionStart=0, runId=0;
const $=id=>document.getElementById(id);
const esc=x=>String(x).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
function shuffle(a){ a=a.slice(); for(let k=a.length-1;k>0;k--){const j=Math.floor(Math.random()*(k+1)); [a[k],a[j]]=[a[j],a[k]];} return a; }
function showScreen(n){
  ['Upload','Study','Quiz','Summary'].forEach(k=>$('scr'+k).classList.toggle('hidden',k!==n));
  const step=n==='Summary'?'Quiz':n;
  document.querySelectorAll('.steps span').forEach(e=>e.classList.toggle('on',e.dataset.s===step));
  window.scrollTo({top:0,behavior:'smooth'});
}
/* theme + saved deck (storage can be blocked, so always guarded) */
function store(k,v){ try{ if(v===undefined) return JSON.parse(localStorage.getItem(k)); localStorage.setItem(k,JSON.stringify(v)); }catch(e){ return null; } }
(function(){ const t=store('sb_theme'); if(t) document.documentElement.dataset.theme=t; })();
$('themeBtn').onclick=()=>{
  const dark=document.documentElement.dataset.theme==='dark' || (!document.documentElement.dataset.theme && matchMedia('(prefers-color-scheme:dark)').matches);
  const t=dark?'light':'dark'; document.documentElement.dataset.theme=t; store('sb_theme',t);
};
function saveDeck(){ store('sb_deck',{cards:allCards,known:[...known]}); }
(function(){ const d=store('sb_deck'); if(d&&d.cards&&d.cards.length){ const b=$('resumeBtn'); b.textContent='↩ Resume last deck ('+d.cards.length+' cards)'; b.classList.remove('hidden'); } })();
function resumeDeck(){ const d=store('sb_deck'); if(!d) return; allCards=d.cards; known=new Set(d.known||[]); setFilter('all',true); showScreen('Study'); }

/* ---------- upload ---------- */
function setFile(f){
  if(!f) return;
  const dt=new DataTransfer(); dt.items.add(f); $('pdf').files=dt.files;
  $('fname').textContent=f.name; $('fmeta').textContent=(f.size/1048576).toFixed(1)+' MB';
}
$('pdf').addEventListener('change',()=>{ const f=$('pdf').files[0]; if(f) setFile(f); });
const drop=$('drop');
['dragenter','dragover'].forEach(e=>drop.addEventListener(e,ev=>{ev.preventDefault(); drop.classList.add('over');}));
['dragleave','drop'].forEach(e=>drop.addEventListener(e,ev=>{ev.preventDefault(); drop.classList.remove('over');}));
drop.addEventListener('drop',ev=>{ const f=ev.dataTransfer.files[0]; if(f&&/pdf$/i.test(f.name)) setFile(f); else $('status').textContent='Please drop a PDF file.'; });
$('target').addEventListener('input',()=>{ $('targetVal').textContent=$('target').value; });

async function makeCards(){
  const f=$('pdf').files[0];
  if(!f){ $('status').textContent='Pick a PDF first 👆'; return; }
  if(f.size>50*1024*1024){ $('status').textContent='😕 That PDF is over 20 MB.'; return; }
  const tgt=parseInt($('target').value)||30;
  if(tgt>50 && !confirm('Making '+tgt+' cards can take a while. Continue?')) return;
  $('makeBtn').disabled=true; $('genBar').classList.remove('hidden'); $('genFill').style.width='4%';
  $('status').innerHTML='<span class="spin">🧠</span> Reading your PDF…';
  const fd=new FormData(); fd.append('pdf',f); fd.append('target',tgt);
  fd.append('topic',$('topic').value); fd.append('start',$('pgStart').value); fd.append('end',$('pgEnd').value);
  try{
    const d=await (await fetch('/upload',{method:'POST',body:fd})).json();
    if(d.error){ $('status').textContent='😕 '+d.error; return; }
    while(true){
      await new Promise(r=>setTimeout(r,1500));
      const j=await (await fetch('/job/'+d.job_id)).json();
      if(j.status==='error'){ $('status').textContent='😕 '+j.error; return; }
      if(j.status==='done'){
        if(!j.cards.length){ $('status').textContent='😕 No cards came out. Try another PDF, topic or page range.'; return; }
        allCards=j.cards; known=new Set(); saveDeck(); setFilter('all',true);
        $('status').textContent=''; showScreen('Study'); return;
      }
      if(j.total){ $('genFill').style.width=Math.max(6,j.done/j.total*100)+'%';
        $('status').innerHTML='<span class="spin">🧠</span> Section '+j.done+' of '+j.total+' · '+j.found+' cards so far'; }
    }
  }catch(e){ $('status').textContent='😕 Request failed. Check the notebook output for errors.'; }
  finally{ $('makeBtn').disabled=false; $('genBar').classList.add('hidden'); $('genFill').style.width='0'; }
}
function exportCards(){
  const clean=x=>String(x).replace(/[\t\r\n]+/g,' ');
  const tsv=allCards.map(c=>clean(c.q)+'\t'+clean(c.a)).join('\n');
  const a=document.createElement('a');
  a.href=URL.createObjectURL(new Blob([tsv],{type:'text/tab-separated-values'}));
  a.download='study_buddy_cards.tsv'; a.click();
}

/* ---------- study mode ---------- */
function setFilter(f,reset){
  filter=f; $('fAll').classList.toggle('on',f==='all'); $('fLearn').classList.toggle('on',f==='learn');
  studyCards = f==='learn' ? allCards.filter(c=>!known.has(c.q)) : allCards.slice();
  if(!studyCards.length){ studyCards=allCards.slice(); filter='all'; $('fAll').classList.add('on'); $('fLearn').classList.remove('on'); }
  s=0; renderStudy();
}
function renderStudy(){
  const wasFlipped=$('flipper').classList.contains('flipped');
  $('flipper').classList.remove('flipped');
  $('sPos').textContent='Card '+(s+1)+' / '+studyCards.length;
  $('knownPill').textContent='✓ '+known.size+' / '+allCards.length+' known';
  $('studyBar').style.width=(allCards.length?known.size/allCards.length*100:0)+'%';
  $('sQ').textContent=studyCards[s].q;
  setTimeout(()=>{ $('sA').textContent=studyCards[s].a; }, wasFlipped?280:0);
}
function flipCard(){ $('flipper').classList.toggle('flipped'); }
function studyMove(d){ s=(s+d+studyCards.length)%studyCards.length; renderStudy(); }
function shuffleStudy(){ studyCards=shuffle(studyCards); s=0; renderStudy(); }
function mark(ok){
  const q=studyCards[s].q; ok?known.add(q):known.delete(q); saveDeck();
  if(filter==='learn' && ok){ studyCards.splice(s,1); if(!studyCards.length){ setFilter('all'); return; } if(s>=studyCards.length) s=0; renderStudy(); }
  else studyMove(1);
}
function setLimit(n){ $('limit').value=n; }

/* ---------- quiz ---------- */
function startQuiz(){ limit=Math.max(0,parseInt($('limit').value)||0); runTest(allCards); }
function runTest(pool){
  runId++; clearInterval(tick); clearTimeout(nextTimer);
  cards=shuffle(pool); i=0; results=[]; streak=0; bestStreak=0; correctCount=0;
  busy=false; awaitingNext=false; sessionStart=Date.now();
  $('timerTrack').parentElement.style.visibility = limit>0 ? 'visible' : 'hidden';
  showScreen('Quiz'); showQuestion();
}
function updatePills(){
  $('scorePill').textContent='✅ '+correctCount;
  $('missPill').textContent='❌ '+(results.length-correctCount);
  $('streakPill').textContent='🔥 '+streak;
}
function showQuestion(){
  busy=false; awaitingNext=false;
  $('qPos').textContent='Question '+(i+1)+' / '+cards.length;
  $('qText').textContent=cards[i].q;
  $('answer').value=''; $('answer').disabled=false; $('submitBtn').disabled=false;
  $('feedback').classList.add('hidden');
  updatePills();
  $('progress').style.width=(i/cards.length*100)+'%';
  $('answer').focus();
  t0=Date.now(); clearInterval(tick);
  tick=setInterval(onTick,100); onTick();
}
function onTick(){
  const el=(Date.now()-t0)/1000;
  if(limit>0){
    const rem=Math.max(0,limit-el);
    $('timerText').textContent='⏱ '+Math.ceil(rem)+'s';
    $('timerBar').style.width=(rem/limit*100)+'%';
    $('timerTrack').classList.toggle('low', rem<=Math.min(5,limit*0.25));
    if(rem<=0) submitAnswer(true);
  } else { $('timerText').textContent='⏱ '+Math.floor(el)+'s'; }
}
async function submitAnswer(timedOut){
  if(busy) return; busy=true;
  const myRun=runId;
  clearInterval(tick);
  let secs=(Date.now()-t0)/1000; if(limit>0) secs=Math.min(secs,limit);
  const ans=$('answer').value; $('answer').disabled=true; $('submitBtn').disabled=true;
  const card=cards[i];
  $('feedback').className='feedback'; $('feedback').innerHTML='<span class="spin">⚖️</span> Grading…';
  let ok=false, score=0;
  try{
    const g=await (await fetch('/grade',{method:'POST',headers:{'Content-Type':'application/json'},
      body:JSON.stringify({answer:ans,reference:card.a,question:card.q})})).json();
    ok=!!g.ok; score=g.score||0;
  }catch(e){ ok=false; }
  if(myRun!==runId) return;
  results.push({q:card.q,a:card.a,secs,ok});
  if(ok){ correctCount++; streak++; bestStreak=Math.max(bestStreak,streak); } else { streak=0; }
  updatePills();
  $('progress').style.width=((i+1)/cards.length*100)+'%';
  const head = ok ? '🎉 Correct!' : (timedOut ? "⏰ Time's up" : '❌ Not quite');
  const fb=$('feedback'); fb.className='feedback '+(ok?'good':'bad');
  fb.innerHTML='<b>'+head+'</b> <span class="muted small">· '+secs.toFixed(1)+'s · match '+Math.round(score*100)+'%</span>'+
    (ok?'':'<div class="cmp"><b>Your answer</b>'+(ans.trim()?esc(ans):'<i>(no answer)</i>')+'</div>')+
    '<div class="cmp"><b>Correct answer</b>'+esc(card.a)+'</div>'+
    '<button class="'+(ok?'good':'')+'" onclick="advance()">'+(i+1<cards.length?'Next →':'See results')+'</button>';
  awaitingNext=true;
  if(ok) nextTimer=setTimeout(advance,1800);   // wrong answers wait for you to read the correction
}
function advance(){
  if(!awaitingNext) return;
  awaitingNext=false; clearTimeout(nextTimer);
  i++; if(i<cards.length) showQuestion(); else finish(false);
}
function exitTest(){
  runId++; clearInterval(tick); clearTimeout(nextTimer); busy=false; awaitingNext=false;
  if(results.length>0) finish(true); else showScreen('Study');
}

/* ---------- summary ---------- */
function finish(early){
  clearInterval(tick); clearTimeout(nextTimer);
  const n=results.length, correct=results.filter(r=>r.ok).length;
  const avg=results.reduce((a,r)=>a+r.secs,0)/n, total=(Date.now()-sessionStart)/1000, pct=correct/n;
  missedCards=results.filter(r=>!r.ok).map(r=>({q:r.q,a:r.a}));
  $('sumTitle').textContent = pct===1?'🏆 Perfect run!':pct>=0.8?'🎉 Great job!':pct>=0.5?'💪 Good effort!':'📚 Keep practising!';
  $('sumNote').textContent = early ? ('Exited early: '+n+' of '+cards.length+' answered') : '';
  $('ringNum').textContent=Math.round(pct*100)+'%';
  $('ringFg').style.stroke = pct>=0.8?'var(--good)':pct>=0.5?'var(--primary)':'var(--bad)';
  $('ringFg').style.strokeDashoffset=364.4;
  setTimeout(()=>{ $('ringFg').style.strokeDashoffset=364.4*(1-pct); },60);
  $('stScore').textContent=correct+'/'+n;
  $('stAvg').textContent=avg.toFixed(1)+'s';
  $('stTotal').textContent=Math.round(total)+'s';
  $('stStreak').textContent='🔥 '+bestStreak;
  $('missedBox').innerHTML = missedCards.length
    ? '<h3>Revise these</h3><ul class="missed">'+missedCards.map(m=>'<li><b>'+esc(m.q)+'</b><br><span>'+esc(m.a)+'</span></li>').join('')+'</ul>'
    : '<h3 style="text-align:center">No misses 🎉</h3>';
  $('sumButtons').innerHTML =
    (missedCards.length ? '<button class="bad" onclick="retakeMissed()">🎯 Retake missed ('+missedCards.length+')</button>' : '')+
    '<button onclick="retakeFull()">🔁 Retake test</button>'+
    '<button class="ghost" onclick="backToStudy()">📚 Back to cards</button>';
  showScreen('Summary');
}
function retakeFull(){ runTest(allCards); }
function retakeMissed(){ runTest(missedCards); }
function backToStudy(){ setFilter('all'); showScreen('Study'); }

/* ---------- keyboard ---------- */
document.addEventListener('keydown',e=>{
  const tag=document.activeElement.tagName;
  if(!$('scrStudy').classList.contains('hidden') && tag!=='INPUT'){
    if(e.key==='ArrowRight') studyMove(1);
    else if(e.key==='ArrowLeft') studyMove(-1);
    else if(e.key===' '){ e.preventDefault(); flipCard(); }
    else if(e.key==='1') mark(false);
    else if(e.key==='2') mark(true);
  }
  if(!$('scrQuiz').classList.contains('hidden') && e.key==='Enter' && tag!=='BUTTON'){
    e.preventDefault();
    awaitingNext ? advance() : submitAnswer(false);
  }
});
showScreen('Upload');
</script><script>
/* a different gradient for every flashcard */
const PALS=[["#7c3aed","#ec4899"],["#f97316","#ef4444"],["#06b6d4","#3b82f6"],["#10b981","#84cc16"],
            ["#a855f7","#6366f1"],["#f59e0b","#ec4899"],["#14b8a6","#6366f1"],["#ef4444","#8b5cf6"]];
const grad=p=>'linear-gradient(135deg,'+p[0]+','+p[1]+')';
const _renderStudy=renderStudy;
renderStudy=function(){
  _renderStudy();
  document.querySelector('.front').style.background=grad(PALS[s%PALS.length]);
  document.querySelector('.back').style.background=grad(PALS[(s+3)%PALS.length]);
};
/* confetti for a good score */
function confetti(){
  const em=['🎉','✨','🎊','⭐','💜','💙','💚','🧡'];
  for(let k=0;k<45;k++){
    const e=document.createElement('div');
    e.textContent=em[k%em.length];
    e.style.cssText='position:fixed;top:-30px;left:'+Math.random()*100+'vw;font-size:'+(16+Math.random()*18)+'px;z-index:99;pointer-events:none;'+
      'animation:fall '+(2.2+Math.random()*2)+'s linear '+Math.random()*0.8+'s forwards';
    document.body.appendChild(e); setTimeout(()=>e.remove(),5500);
  }
}
const _finish=finish;
finish=function(early){
  _finish(early);
  const n=results.length; if(n && results.filter(r=>r.ok).length/n>=0.8) confetti();
};
if(!$('scrStudy').classList.contains('hidden') && studyCards.length) renderStudy();
</script></body></html>"""
print("UI loaded. Now run the next cell to (re)start the server, then refresh the page.")


In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from werkzeug.serving import make_server
import threading

app = Flask(__name__)

assert "PAGE" in globals(), "Run the UI cell (the one that defines PAGE) before this one."

@app.route("/")
def home():
    return PAGE   # returned as-is (no Jinja), so curly braces in the JS/CSS are safe

app.config["MAX_CONTENT_LENGTH"] = (MAX_MB + 1) * 1024 * 1024

@app.errorhandler(413)
def too_big(e):
    return jsonify(error=f"PDF is larger than {MAX_MB} MB."), 413

def _int_or_none(v):
    try: return int(v)
    except (TypeError, ValueError): return None

def run_job(job_id, file_bytes, target, start, end, topic):
    job = JOBS[job_id]
    try:
        doc_id, n = ingest(file_bytes, start, end)
        job.update(doc_id=doc_id, total=0)
        cards = generate_cards(doc_id, target, topic, job)
        job.update(status="done", cards=cards)
    except Exception as e:
        job.update(status="error", error=str(e))

@app.route("/upload", methods=["POST"])
def upload():
    try:
        target = max(1, min(int(request.form.get("target", 30)), 100))
        file_bytes = request.files["pdf"].read()
    except Exception as e:
        return jsonify(error=str(e)), 400
    job_id = uuid.uuid4().hex[:8]
    JOBS[job_id] = {"status": "running", "done": 0, "total": 0, "found": 0}
    threading.Thread(target=run_job, daemon=True, args=(
        job_id, file_bytes, target, _int_or_none(request.form.get("start")),
        _int_or_none(request.form.get("end")), request.form.get("topic", ""))).start()
    return jsonify(job_id=job_id)

@app.route("/job/<job_id>")
def job_status(job_id):
    job = JOBS.get(job_id)
    return jsonify(job) if job else (jsonify(status="error", error="Unknown job"), 404)

@app.route("/grade", methods=["POST"])
def grade_route():
    d = request.json
    score, ok = grade(d["answer"], d["reference"], d.get("question", ""))
    return jsonify(score=score, ok=ok)

# Stop the previous server if this cell was run before
try:
    server.shutdown()
except NameError:
    pass

ngrok.kill()
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
print("Open:", ngrok.connect(5000).public_url)
print("If ngrok shows a 'Visit Site' warning page, click it once. That's ngrok's free-tier interstitial.")

server = make_server("0.0.0.0", 5000, app)
threading.Thread(target=server.serve_forever, daemon=True).start()

In [ ]:
print(grade("It can solve problems that are too complex to write rules for by hand.",
            "Machine learning can handle problems where writing rules by hand is impractical."))
# direction check: should be (high similarity, False)
print(grade("Gradient descent increases the loss.", "Gradient descent decreases the loss."))